In [1]:
from selenium import webdriver
from selenium.webdriver import Chrome, ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import  WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import pandas as pd 
import time, threading
import re
import os
import requests

In [45]:
def scarpe(path):
    options = ChromeOptions()
    options.headless=True
    service = Service(ChromeDriverManager().install())
    driver = Chrome(service=service, options=options)
    driver.get(path)
    driver.maximize_window()
    
     # clck login
    try:
        login = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/a[text()="Login"]')))
        if login:
            login.click()
    except Exception as e:
        print("no login")

    # get the username tab
    try:
        provided_u_name = "FOU009"                                                     
        user_name = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID,'username')))   
        user_name.send_keys(provided_u_name)
    except Exception as e:
        print(f"No username tab found and the error is {e}")
    
    # get password tab
    try:
        provided_pass = "fwob!!33"
        password = WebDriverWait(driver, 2).until(EC.presence_of_element_located((By.ID, 'password')))
        password.send_keys(provided_pass)
    except Exception as e:
        print(f"No password tab found and the error is {e}")
    
    # click the submit
    try:
        # accept terms
        terms = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/input[@id="chkTC"]')))
        if terms:
            terms.click()
        else:
            print("No terms found")
        login = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/button[@type="submit"]')))
        if login:
            login.click()
            time.sleep(2)
        else:
            print("No login")
            
        
        for _ in range(2):
            driver.back()
            # time.sleep(1)
    except Exception as e:
        print("No login found")
    
    # view as a grid
    try:
        grid =WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/button[text() = "Display as grid"]'))) 
        if grid:
            grid.click()
    except Exception as e:
        print("No grid")
    
    
    results = []
    car_count = 0
    while True: #cars
        # if not driver.window_handles:
        #     print("Browser window closed. Stopping script.")
        #     break
        try:
            page_cars = WebDriverWait(driver, 5).until(EC.presence_of_all_elements_located((By.XPATH, './/img[@class="img-table"]')))
            
            for i in range(len(page_cars)): # len(page_cars)
                try:
                    page_cars = WebDriverWait(driver, 5).until(EC.presence_of_all_elements_located((By.XPATH, './/img[@class="img-table"]')))
                    # driver.execute_script("arguments[0].scrollIntoView();", page_cars[i])
                    driver.execute_script("arguments[0].scrollIntoView({block: 'nearest'});", page_cars[i])

                    time.sleep(1)
                    page_cars[i].click()
                    time.sleep(2)

                    details = {}
                    
                    # title
                    try:
                        title = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/div[@id="details-header"]'))).text.strip().split("\n")[1]
                        if title:
                            details['Title'] = title
                        else:
                            details['Title'] = 'na'
                    except Exception as e:
                        print("No title")
                    
                    # details
                    try:
                        det_card = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/div[@id="detailsTable"]')))
                        if det_card:
                            det_rows = det_card.find_elements(By.TAG_NAME, 'tr')
                            if det_rows:
                                for row in det_rows:
                                    det_lbl = row.find_element(By.TAG_NAME, 'th').text.strip()
                                    
                                    if det_lbl =="Mileage Warranted":
                                        det_val1 = row.find_element(By.XPATH, '(.//i[@class="icon-check text-success"])[1]')
                                        if det_val1:
                                            details[det_lbl] ='Warranted' 
                                        else:
                                            details[det_lbl] = "Not Warranted"

                                    elif det_lbl =="V5":
                                        det_val2 = row.find_element(By.XPATH, './/i[@class="icon-check text-success v5"]')
                                        if det_val2:
                                            details[det_lbl] = "Warranted"
                                        else:
                                            details[det_lbl] = "Follow on"
                                            
                                    elif det_lbl == "Imported":
                                        det_val3 = row.find_element(By.XPATH, './/td[@ng-show="!Listing.Imported"]')
                                        if det_val3:
                                            details[det_lbl] = "Not Warranted"
                                        else:
                                            details[det_lbl] = 'Warranted'
                                    else:
                                        det_val = row.find_element(By.TAG_NAME, 'td').text.strip()
                                        
                                        if det_lbl and det_val:
                                            details[det_lbl] = det_val
                    except Exception as e:
                        print("No det card")
                    
                    # images
                    try:
                        img_click = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/div[@id="viewImages"]')))
                        if img_click:
                            driver.execute_script("arguments[0].scrollIntoView({block: 'nearest'});", img_click)
                            img_click.click()
                            time.sleep(1)

                            img_ul = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/ul[@id="aos-primary-slider-list"]')))
                            if img_ul:
                                img_lis =WebDriverWait(img_ul, 5).until(EC.presence_of_all_elements_located(((By.TAG_NAME, 'li'))))
                                if img_lis:
                                    imgs = []
                                    for li in img_lis:
                                        img = li.find_element(By.TAG_NAME, "img").get_attribute("src")
                                        imgs.append(img)
                                    # imgs = [img.get_attribute("src") for img in img_lis]
                                    
                                    details['Images'] = ", ".join(imgs)
                                    # Assuming driver is your active WebDriver instance
                                    ActionChains(driver).send_keys(Keys.ESCAPE).perform()

                    except Exception as e:
                        print("No images")
                    
                    # inspection 
                    try:
                        inspec = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/th[@id="inspection-link"]/a'))).get_attribute("href")
                        if inspec:
                            details['Inspection'] = inspec
                        else:
                            details['Inspection'] = "na"
                    except Exception as e:
                        print("No inspcetion")
                    
                    results.append(details)
                    car_count+=1
                    driver.back()
                except Exception as e:
                    print("No more cars found")
                    
            if car_count % 24 == 0:
                try:
                    next_link = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, './/a[contains(@aria-label,"Next") and not(contains(@disabled,"disabled"))]')))
                    if next_link:
                        # driver.execute_script("arguments[0].scrollIntoView();", next_link)
                        time.sleep(2)
                        next_link.click()
                        
                        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, './/img[@class="img-table"]')))
                        driver.refresh()
                        print("Hit the next button")
                except Exception as e:
                    print("No next found")
                    break
        except Exception as e:
            print("No more cars to click")
            break
    df = pd.DataFrame.from_dict(results)
    df.to_csv("cva.csv", index=False)
    # time.sleep(5)
    driver.quit()

path = "https://www.cva-auctions.co.uk/stock?saleid=566"
scarpe(path)

No det card
No det card
No images
No det card
No images
No det card
No det card
No det card
No det card
No det card
No det card
No det card
No det card
No det card
No det card
No more cars found
No more cars found
No more cars found
No more cars found
No more cars found
No more cars found
No more cars found
No more cars found
No more cars found
No more cars found
No more cars found
No more cars to click


In [2]:
df=pd.read_csv("cva.csv")
df.columns

Index(['Unnamed: 0', 'Title', 'Registration Number', 'Date of Registration',
       'Manufacture Date', 'Make', 'Model', 'Body Type', 'Mileage',
       'Transmission', 'Fuel Type', 'MOT Expiry', 'Service History',
       'Lot Number', 'VAT Status', 'Former Keepers', 'Vendor', 'Sale Date',
       'Location', 'Remarks', 'Images'],
      dtype='object')

In [3]:
from urllib.parse import urlparse, urljoin
# urlparse: Parses a URL into components (scheme, netloc, path, etc.), making it easy to validate or extract information from the URL.
# urljoin: Joins a base URL with a relative path to form a complete URL.
df = pd.read_csv("cva.csv")

reg_img = df[["Registration Number", "Images"]]
inspec = df[["Registration Number", 'Inspection']]

# images
def download_images(data, main_folder="Images"): 
    os.makedirs(main_folder, exist_ok=True)
    
    for index, row in data.iterrows():
        reg_no = row["Registration Number"] 
        image_urls = row["Images"]

        # Check if 'Images' column is empty or NaN
        if pd.isna(reg_no) or pd.isna(image_urls) or not isinstance(image_urls, str) or image_urls.strip() == "":
            print(f"Skipping {index} (No image URLs)")
            continue

        image_urls = image_urls.split(", ")  # Split URLs by comma
        
        reg_folder = os.path.join(main_folder, reg_no)
        os.makedirs(reg_folder, exist_ok=True)
        
        for idx, url in enumerate(image_urls):
            url = url.strip()
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url) 
            
            parsed_url = urlparse(url)
            if not parsed_url.scheme or not parsed_url.netloc:
                print(f"Invalid URL skipped: {url}") 
                continue
            
            try:
                response = requests.get(url, stream=True) 
                response.raise_for_status()
                
                file_name = os.path.basename(parsed_url.path) or f"image_{idx + 1}.jpg"
                file_extension = file_name.split(".")[-1]
                
                if file_extension not in ["jpg", "jpeg", "png", "gif", "bmp", "webp"]:
                    file_name = f"image_{idx + 1}.jpg"
                
                full_file_name = os.path.join(reg_folder, f"{reg_no}_{idx + 1}.jpg")
                
                with open(full_file_name, 'wb') as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)
                
                print(f"Downloaded: {full_file_name}")
            except Exception as e:
                print(f"Failed to download {url} for {reg_no}: {e}")

# report
def download_repos(data, main_folder="report"): 
    os.makedirs(main_folder, exist_ok=True)
    
    for index, row in data.iterrows():
        reg_no = row["Registration Number"] 
        repo_urls = row["Inspection"]

        # Check if 'Images' column is empty or NaN
        if pd.isna(reg_no) or pd.isna(repo_urls) or not isinstance(repo_urls, str) or repo_urls.strip() == "":
            print(f"Skipping {index} (No image URLs)")
            continue

        repo_urls = repo_urls.split(", ")  # Split URLs by comma
        
        # reg_folder = os.path.join(main_folder, reg_no)
        # os.makedirs(reg_folder, exist_ok=True)
        
        for idx, url in enumerate(repo_urls):
            url = url.strip()
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url) 
            
            parsed_url = urlparse(url)
            if not parsed_url.scheme or not parsed_url.netloc:
                print(f"Invalid URL skipped: {url}") 
                continue
            
            try:
                response = requests.get(url, stream=True) 
                response.raise_for_status()
                
                file_name = os.path.basename(parsed_url.path) or f"image_{idx + 1}.pdf"
                file_extension = file_name.split(".")[-1]
                
                if file_extension not in ["pdf", "jpeg", "png", "gif", "bmp", "webp"]:
                    file_name = f"image_{idx + 1}.pdf"
                
                full_file_name = os.path.join(main_folder, f"{reg_no}.pdf")
                
                with open(full_file_name, 'wb') as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)
                
                print(f"Downloaded: {full_file_name}")
            except Exception as e:
                print(f"Failed to download {url} for {reg_no}: {e}")

def start_funcs():
    thread1 = threading.Thread(target=download_images, args=(reg_img, ))
    thread2 = threading.Thread(target=download_repos, args=(inspec, ))
    
    thread1.start()
    thread2.start()
    
    thread1.join()
    thread2.join()
    
if __name__ =="__main__":
    start_funcs()

Downloaded: Images\PF20HFE\PF20HFE_1.jpg
Downloaded: Images\PF20HFE\PF20HFE_2.jpg
Downloaded: report\PF20HFE.pdf
Downloaded: Images\PF20HFE\PF20HFE_3.jpg
Downloaded: Images\PF20HFE\PF20HFE_4.jpg
Downloaded: report\PJ20FVR.pdf
Downloaded: Images\PF20HFE\PF20HFE_5.jpg
Downloaded: Images\PF20HFE\PF20HFE_6.jpg
Downloaded: Images\PF20HFE\PF20HFE_7.jpg
Downloaded: report\PJ20PTX.pdf
Downloaded: Images\PF20HFE\PF20HFE_8.jpg
Downloaded: Images\PF20HFE\PF20HFE_9.jpg
Downloaded: Images\PF20HFE\PF20HFE_10.jpg
Downloaded: report\MA21GPX.pdf
Invalid URL skipped: https:///na
Downloaded: Images\PF20HFE\PF20HFE_11.jpg
Downloaded: Images\PF20HFE\PF20HFE_12.jpg
Downloaded: report\MT21OCA.pdf
Downloaded: Images\PF20HFE\PF20HFE_13.jpg
Downloaded: Images\PF20HFE\PF20HFE_14.jpg
Downloaded: report\DL70NVY.pdf
Downloaded: Images\PF20HFE\PF20HFE_15.jpg
Downloaded: Images\PF20HFE\PF20HFE_16.jpg
Downloaded: report\MK71PNE.pdf
Invalid URL skipped: https:///na
Downloaded: Images\PF20HFE\PF20HFE_17.jpg
Skipping 1 (